# Predicting Vehicle Prices Using Regression Models

## **Objectives**  
The primary goal of this project is to develop a robust regression model to predict used car prices for a reseller based on various listed features and specifications. In addition to predicting prices, the project focuses on identifying feature importance and mitigating overfitting through the application of regularisation techniques.

There can be several business objectives for this, such as:

* **Price Prediction**: Model car prices based on features like mileage, fuel type, and performance.
* **Market Analysis**: Explore trends and preferences in the used car market, by type, region, or other metrics.
* **Feature Importance**: Identify the most important factors influencing car prices (e.g., fuel type, mileage, age).

### Workflow Overview

The analysis follows a reproducible regression workflow:

1. Understand the dataset and assess data quality.
2. Explore distributions, relationships, and potential outliers.
3. Engineer and encode features, then prepare training and test sets.
4. Train a baseline linear regression model.
5. Tune Ridge and Lasso regularisation and compare model performance.
6. Summarise findings, limitations, and practical implications.

## **1 Data Understanding**

| **Variable** | **Description** |
--------|--------------|
| `make_model` | The brand and model of the vehicle (e.g., 'Audi A1'). |
| `body_type` | The body style of the vehicle, such as Sedan, Compact, or Station Wagon. |
| `price`  | The listed price of the car in currency. |
| `vat`  | Indicates the VAT status for the vehicle's price (e.g., VAT deductible, Price negotiable). |
| `km` | The total mileage (in kilometers) of the vehicle, indicating its usage. |
| `Type` | Condition of the vehicle, whether it's 'Used' or 'New'.|
| `Fuel` | Type of fuel the vehicle uses, such as 'Diesel', 'Benzine', etc. |
| `Gears` | The number of gears in the vehicle's transmission. |
| `Comfort_Convenience` | Comfort and convenience features, such as 'Air conditioning', 'Leather steering wheel', 'Cruise control', and more. |
| `Entertainment_Media` | Media features available in the vehicle, including 'Bluetooth', 'MP3', 'Radio', etc. |
| `Extras` | Additional features like 'Alloy wheels', 'Sport suspension', etc.|
| `Safety_Security` | Safety features like 'ABS', 'Airbags', 'Electronic stability control', 'Isofix', etc.  |
| `age` | Age of the car (calculated based on the model year). |
| `Previous_Owners`| The number of previous owners the car has had. |
| `hp_kW` | Engine power in kilowatts (kW), indicating the performance capacity of the engine.|
| `Inspection_new` | Indicates whether the car has recently undergone an inspection (1 for yes, 0 for no). |
| `Paint_Type`| The type of paint on the car, such as 'Metallic', 'Matte', etc. |
| `Upholstery_type` | The material used for the interior upholstery, such as 'Cloth', 'Leather', etc.|
| `Gearing_Type` | The type of transmission the car uses, either 'Automatic' or 'Manual'. |
| `Displacement_cc` | The engine displacement in cubic centimeters (cc), indicating the size of the engine.|
| `Weight_kg` | The total weight of the vehicle in kilograms.|
| `Drive_chain` | The type of drivetrain, indicating whether it's 'Front' or 'Rear' wheel drive. |
| `cons_comb`  | The combined fuel consumption in liters per 100 kilometers.|

### **1.1 Data Loading**

**Importing Necessary Libraries**

In [ ]:
# Importing necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)


#### **1.1.1**
Load the dataset

In [ ]:
# Load the data
df = pd.read_csv('Car_Price_data.csv')

In [ ]:
pd.set_option("display.max_columns", None)

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.describe().round(2)

In [ ]:
df.info()

In [ ]:
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
)

## **2. Data Analysis and Feature Engineering**

### **2.1 Preliminary Analysis and Feature Distributions**

#### **2.1.1 Missing-Value Assessment**

Inspect and handle missing values.

In [ ]:
# Find the proportion of missing values in each column and handle if found
# Find the proportion of missing values in each column and handle if found
missing_pct = df.isnull().mean().sort_values(ascending=False) * 100
missing_pct = missing_pct[missing_pct > 0]
print(missing_pct)

In [ ]:
df.duplicated().sum()

In [ ]:
df = df.drop_duplicates().copy()

In [ ]:
df["price"].isna().sum()

In [ ]:
df["price"].isnull().sum()

In [ ]:
df["price"].eq(0).sum()

**From the features, identify the target feature and numerical and categorical predictors. Select the numerical and categorical features carefully as they will be used in analysis.**

#### **2.1.2 Numerical Feature Distributions**

Identify numerical predictors and visualise their distributions.

In [ ]:
# Identify numerical features and plot histograms
target = 'price'
numerical_features = ['km', 'age', 'hp_kw', 'gears', 'previous_owners',
                       'displacement_cc', 'weight_kg', 'cons_comb']
numerical_features = [c for c in numerical_features if c in df.columns]

df[numerical_features].hist(figsize=(16, 10), bins=30)
plt.suptitle('Frequency Distribution of Numerical Predictors')
plt.tight_layout()
plt.show()

#### **2.1.3 Categorical Feature Distributions**

Identify categorical predictors and visualise their distributions.

In [ ]:
# Identify categorical columns and check their frequency distributions
# Identify categorical columns and check their frequency distributions
categorical_features = ['body_type', 'vat', 'type', 'fuel', 'inspection_new',
                         'paint_type', 'upholstery_type', 'gearing_type', 'drive_chain']
categorical_features = [c for c in categorical_features if c in df.columns]

fig, axes = plt.subplots(nrows=(len(categorical_features)+2)//3, ncols=3, figsize=(18, 12))
axes = axes.flatten()
for i, col in enumerate(categorical_features):
    df[col].value_counts().plot(kind='bar', ax=axes[i])
    axes[i].set_title(col)
for j in range(i+1, len(axes)):
    axes[j].axis('off')
plt.tight_layout()
plt.show()

for col in categorical_features:
    print(f'--- {col} ---')
    print(df[col].value_counts(normalize=True) * 100)
    print()


**Note**: Look carefully at the values stored in columns `["Comfort_Convenience", "Entertainment_Media", "Extras", "Safety_Security"]`.

Should they be considered categorical? Should they be dropped or handled any other way?

#### **2.1.4 Rare Categories and Class Imbalance**

Review low-frequency categories and address them where appropriate.

Some information regarding values in the `Type` column that may help:
- *'Pre-registered'* cars are ones which have already been registered previously by the seller.
- *'New'* cars are not necessarily new cars, but new-like cars. These might also have multiple owners due to multiple pre-registrations as well.
- *'Employee's car'* are cars used by employees over a short period of time and small distance.
- *'Demonstration'* cars are used for trial purposes and also driven for a short time and distance.

Based on these, you can handle this particular column. For other columns, decide a strategy on your own.

In [ ]:
# Fix columns as needed
if 'Type' in df.columns:
    print(df['Type'].value_counts())
    type_map = {
        'Used': 'Used',
        'New': 'New',
        'Pre-registered': 'Pre-registered',
        "Employee's car": 'Demonstration',
        'Demonstration': 'Demonstration',
        'Antique': 'New'
    }
    df['Type'] = df['Type'].map(lambda x: type_map.get(x, x))
    print(df['Type'].value_counts())

# For other categorical columns: merge categories below a frequency threshold into 'Other'
threshold = 0.02  # 2%
for col in categorical_features:
    freq = df[col].value_counts(normalize=True)
    rare_labels = freq[freq < threshold].index
    if len(rare_labels) > 0:
        df[col] = df[col].replace(rare_labels, 'Other')

for col in categorical_features:
    print(f'--- {col} after fix ---')
    print(df[col].value_counts(normalize=True) * 100)
    print()



#### **2.1.5 Target Variable Distribution and Transformation**

Examine the target variable and apply a suitable transformation if needed.

In [ ]:
# Plot histograms for target feature
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['price'], kde=True, ax=axes[0])
axes[0].set_title('Price Distribution')
stats.probplot(df['price'], dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot: Price')
plt.tight_layout()
plt.show()

print('Skewness:', df['price'].skew())


**The target variable seems to be skewed. Perform suitable transformation on the target.**

In [ ]:
# Transform the target feature
df['log_price'] = np.log1p(df['price'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['log_price'], kde=True, ax=axes[0])
axes[0].set_title('Log(Price) Distribution')
stats.probplot(df['log_price'], dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot: Log(Price)')
plt.tight_layout()
plt.show()

print('Skewness after log transform:', df['log_price'].skew())


### **2.2 Correlation Analysis**

#### **2.2.1 Numerical Feature Correlations**

Visualise correlations between numerical features and the target variable.

In [ ]:
# Visualise correlation
corr_cols = numerical_features + ['price', 'log_price']
corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap: Numerical Features vs Target')
plt.tight_layout()
plt.show()


#### **2.2.2 Categorical Feature Relationships**

Analyse relationships between categorical features and the target variable.

In [ ]:
# Comparing average values of target for different categories
for col in categorical_features:
    plt.figure(figsize=(8, 4))
    order = df.groupby(col)['log_price'].mean().sort_values(ascending=False).index
    sns.boxplot(data=df, x=col, y='log_price', order=order)
    plt.title(f'log_price by {col}')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    print(df.groupby(col)['price'].mean().sort_values(ascending=False))
    print()


### **2.3 Outlier Analysis**

#### **2.3.1 Outlier Detection**

Identify potential outliers in the dataset.

In [ ]:
# Outliers present in each column
plt.figure(figsize=(16, 8))
for i, col in enumerate(numerical_features + ['price'], 1):
    plt.subplot(3, 3, i)
    sns.boxplot(y=df[col])
    plt.title(col)
plt.tight_layout()
plt.show()

# IQR-based outlier count
for col in numerical_features + ['price']:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f'{col}: {n_outliers} outliers ({n_outliers/len(df)*100:.2f}%)')


#### **2.3.2 Outlier Treatment**

Assess and handle outliers using an appropriate approach.

In [ ]:
# Handle outliers
cols_to_cap = ['km', 'hp_kw', 'displacement_cc', 'weight_kg', 'cons_comb', 'price']
cols_to_cap = [c for c in cols_to_cap if c in df.columns]

for col in cols_to_cap:
    lower = df[col].quantile(0.01)
    upper = df[col].quantile(0.99)
    df[col] = df[col].clip(lower, upper)

# Recompute log target after capping
df['log_price'] = np.log1p(df['price'])
print(df[cols_to_cap].describe())

### **2.4 Feature Engineering**

#### **2.4.1**
Fix any redundant columns and create new ones if needed.

In [ ]:
# Fix/create columns as needed
drop_cols = ['make_model']  # high-cardinality identifier; consider brand extraction instead
df['brand'] = df['make_model'].str.split(' ').str[0] if 'make_model' in df.columns else np.nan

if 'make_model' in df.columns:
    df = df.drop(columns=['make_model'])

# Example engineered feature: power-to-weight ratio
if {'hp_kW', 'Weight_kg'}.issubset(df.columns):
    df['power_to_weight'] = df['hp_kW'] / df['Weight_kg']

print(df.columns.tolist())


#### **2.4.2 Feature Extraction from Equipment Lists**

Analyse and engineer features from `Comfort_Convenience`, `Entertainment_Media`, `Extras`, and `Safety_Security`.

These columns contains lists of features present. Decide on how to include these features in the predictors.

In [ ]:
# Check unique values in each feature spec column
spec_cols = ['comfort_convenience', 'entertainment_media', 'extras', 'safety_security']
spec_cols = [c for c in spec_cols if c in df.columns]

spec_value_counts = {}
for col in spec_cols:
    items = df[col].dropna().str.split(',').explode().str.strip()
    counts = items.value_counts()
    spec_value_counts[col] = counts
    print(f'--- {col}: {len(counts)} unique features ---')
    print(counts.head(10))
    print()


Out of these features, we will check the ones which are present in most of the cars or are absent from most of the cars. These kinds of features can be removed as they just increase the dimensionality without explaining the variance.

In [ ]:
# Drop features from df
n = len(df)
for col in spec_cols:
    counts = spec_value_counts[col]
    keep_items = counts[(counts / n >= 0.05) & (counts / n <= 0.95)].index.tolist()
    for item in keep_items:
        flag_col = f'{col}_{item.replace(" ", "_")}'
        df[flag_col] = df[col].fillna('').apply(lambda x, it=item: int(it in x))
    df = df.drop(columns=[col])

print(df.columns.tolist())



#### **2.4.3 Categorical Feature Encoding**

Encode categorical predictors for modelling.

In [ ]:
# Encode features
target_col = 'log_price'
drop_from_X = ['price', 'log_price']

X = df.drop(columns=[c for c in drop_from_X if c in df.columns])
y = df[target_col]

categorical_features_final = X.select_dtypes(include='object').columns.tolist()
X = pd.get_dummies(X, columns=categorical_features_final, drop_first=True)

print(X.shape)
X.head()


#### **2.4.4 Train-Test Split**

Split the prepared dataset into training and testing sets.

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)


#### **2.4.5 Feature Scaling**

Scale the features where appropriate.

In [ ]:
# Scale features
numeric_cols_to_scale = X_train.select_dtypes(include=np.number).columns.tolist()

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols_to_scale] = scaler.fit_transform(X_train[numeric_cols_to_scale])
X_test_scaled[numeric_cols_to_scale] = scaler.transform(X_test[numeric_cols_to_scale])



## **3. Regression Models**

### **3.1 Baseline Linear Regression**

#### **3.1.1 Model Training and Evaluation**

Fit a baseline linear regression model and evaluate it using suitable metrics.

In [ ]:
# Initialise and train model
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

In [ ]:
# Evaluate the model's performance
def evaluate(model, X_tr, y_tr, X_te, y_te, name='Model'):
    pred_tr = model.predict(X_tr)
    pred_te = model.predict(X_te)
    metrics = {
        'Model': name,
        'Train_R2': r2_score(y_tr, pred_tr),
        'Test_R2': r2_score(y_te, pred_te),
        'Train_MAE': mean_absolute_error(y_tr, pred_tr),
        'Test_MAE': mean_absolute_error(y_te, pred_te),
        'Train_RMSE': np.sqrt(mean_squared_error(y_tr, pred_tr)),
        'Test_RMSE': np.sqrt(mean_squared_error(y_te, pred_te)),
    }
    return metrics

baseline_metrics = evaluate(lr, X_train_scaled, y_train, X_test_scaled, y_test, 'Baseline LR')
baseline_metrics


#### **3.1.2 Residual Diagnostics and Model Assumptions**

Inspect residuals and assess key linear regression assumptions.

Check for linearity by analysing residuals vs predicted values

In [ ]:
# Linearity check: Plot residuals vs fitted values
y_pred_train = lr.predict(X_train_scaled)
residuals = y_train - y_pred_train

plt.figure(figsize=(8, 5))
plt.scatter(y_pred_train, residuals, alpha=0.3)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Fitted values')
plt.ylabel('Residuals')
plt.title('Residuals vs Fitted Values')
plt.show()


Check normality in residual distribution

In [ ]:
# Check the normality of residuals by plotting their distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(residuals, kde=True, ax=axes[0])
axes[0].set_title('Residual Distribution')
stats.probplot(residuals, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot of Residuals')
plt.tight_layout()
plt.show()

shapiro_stat, shapiro_p = stats.shapiro(residuals.sample(min(5000, len(residuals)), random_state=1))
print(f'Shapiro-Wilk p-value: {shapiro_p:.5f}')


Check multicollinearity using Variance Inflation Factor (VIF) and handle features with high VIF.

In [ ]:
X_train_scaled = X_train_scaled.astype(float)
X_test_scaled = X_test_scaled.astype(float)

vif_data = pd.DataFrame()
vif_data['feature'] = X_train_scaled.columns
vif_data['VIF'] = [variance_inflation_factor(X_train_scaled.values, i)
                    for i in range(X_train_scaled.shape[1])]

In [ ]:
# Check for multicollinearity and handle
vif_data = pd.DataFrame()
vif_data['feature'] = X_train_scaled.columns
vif_data['VIF'] = [variance_inflation_factor(X_train_scaled.values, i)
                    for i in range(X_train_scaled.shape[1])]
vif_data = vif_data.sort_values('VIF', ascending=False)
print(vif_data.head(15))

# Iteratively drop features with VIF > 10 (highest first), refitting each time
high_vif_features = vif_data[vif_data['VIF'] > 10]['feature'].tolist()
print(f'\nFeatures with VIF > 10: {high_vif_features}')

X_train_scaled = X_train_scaled.drop(columns=high_vif_features, errors='ignore')
X_test_scaled = X_test_scaled.drop(columns=high_vif_features, errors='ignore')

lr = LinearRegression().fit(X_train_scaled, y_train)
baseline_metrics = evaluate(lr, X_train_scaled, y_train, X_test_scaled, y_test, 'Baseline LR (post-VIF)')
baseline_metrics



### **3.2 Ridge Regression**

#### **3.2.1 Initial Regularisation Search**

Define a coarse set of candidate alpha values.

In [ ]:
# List of alphas to tune for Ridge regularisation
ridge_alphas = [0.001, 0.01, 0.1, 1, 5, 10, 20, 50, 100, 200, 500, 1000]


#### **3.2.2 Coarse Alpha Search**

Evaluate Ridge regression across candidate alpha values and identify a promising range.

In [ ]:
# Applying Ridge regression
from sklearn.model_selection import cross_val_score

ridge_train_scores, ridge_test_scores = [], []
for a in ridge_alphas:
    model = Ridge(alpha=a)
    model.fit(X_train_scaled, y_train)
    train_mae = mean_absolute_error(y_train, model.predict(X_train_scaled))
    test_mae = mean_absolute_error(y_test, model.predict(X_test_scaled))
    ridge_train_scores.append(train_mae)
    ridge_test_scores.append(test_mae)


In [ ]:
# Plot train and test scores against alpha
plt.figure(figsize=(8, 5))
plt.plot(ridge_alphas, ridge_train_scores, marker='o', label='Train MAE')
plt.plot(ridge_alphas, ridge_test_scores, marker='o', label='Test MAE')
plt.xscale('log')
plt.xlabel('Alpha (log scale)')
plt.ylabel('MAE')
plt.title('Ridge: MAE vs Alpha')
plt.legend()
plt.show()


Find the best alpha value.

In [ ]:
# Best alpha value
best_ridge_alpha_coarse = ridge_alphas[int(np.argmin(ridge_test_scores))]
print('Best alpha (coarse):', best_ridge_alpha_coarse)

# Best score (negative MAE)
print('Best test MAE (coarse):', -min([-s for s in ridge_test_scores]))

We will get some best value of alpha above. This however is not the most accurate value but the best value from the given list. Now we have a rough estimate of the range that best alpha falls in. Let us do another iteration over the values in a smaller range.

#### **3.2.3 Alpha Fine-Tuning**

Search a narrower alpha range around the initial best result.

In [ ]:
# Take a smaller range of alpha to test
ridge_alphas_fine = np.linspace(1e-6, 5, 40)


In [ ]:
# Applying Ridge regression
ridge_train_scores_fine, ridge_test_scores_fine = [], []
for a in ridge_alphas_fine:
    model = Ridge(alpha=a)
    model.fit(X_train_scaled, y_train)
    train_mae = mean_absolute_error(y_train, model.predict(X_train_scaled))
    test_mae = mean_absolute_error(y_test, model.predict(X_test_scaled))
    ridge_train_scores_fine.append(train_mae)
    ridge_test_scores_fine.append(test_mae)


Plot the error-alpha graph again and find the actual optimal value for alpha.

In [ ]:
# Plot train and test scores against alpha
plt.figure(figsize=(8, 5))
plt.plot(ridge_alphas_fine, ridge_train_scores_fine, marker='o', label='Train MAE')
plt.plot(ridge_alphas_fine, ridge_test_scores_fine, marker='o', label='Test MAE')
plt.xlabel('Alpha')
plt.ylabel('MAE')
plt.title('Ridge: MAE vs Alpha (fine range)')
plt.legend()
plt.show()

# Best alpha value
best_ridge_alpha = ridge_alphas_fine[int(np.argmin(ridge_test_scores_fine))]
print('Best alpha (fine):', best_ridge_alpha)

# Best score (negative MAE)
print('Best test MAE (fine):', min(ridge_test_scores_fine))


In [ ]:
# Set best alpha for Ridge regression
# Fit the Ridge model to get the coefficients of the fitted model
ridge_final = Ridge(alpha=best_ridge_alpha)
ridge_final.fit(X_train_scaled, y_train)

In [ ]:
# Show the coefficients for each feature
ridge_coefs = pd.Series(ridge_final.coef_, index=X_train_scaled.columns).sort_values()
ridge_coefs

In [ ]:
# Evaluate the Ridge model on the test data
ridge_metrics = evaluate(ridge_final, X_train_scaled, y_train, X_test_scaled, y_test, 'Ridge')
ridge_metrics

### **3.3 Lasso Regression**

#### **3.3.1**
Define a list of random alpha values

In [ ]:
# List of alphas to tune for Lasso regularisation
lasso_alphas = [0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1, 5, 10]

#### **3.3.2 Coarse Alpha Search**

Evaluate Lasso regression across candidate alpha values and identify a promising range.

In [ ]:
# Initialise Lasso regression model
lasso_train_scores, lasso_test_scores = [], []
for a in lasso_alphas:
    model = Lasso(alpha=a, max_iter=10000)
    model.fit(X_train_scaled, y_train)
    train_mae = mean_absolute_error(y_train, model.predict(X_train_scaled))
    test_mae = mean_absolute_error(y_test, model.predict(X_test_scaled))
    lasso_train_scores.append(train_mae)
    lasso_test_scores.append(test_mae)

In [ ]:
# Plot train and test scores against alpha
plt.figure(figsize=(8, 5))
plt.plot(lasso_alphas, lasso_train_scores, marker='o', label='Train MAE')
plt.plot(lasso_alphas, lasso_test_scores, marker='o', label='Test MAE')
plt.xscale('log')
plt.xlabel('Alpha (log scale)')
plt.ylabel('MAE')
plt.title('Lasso: MAE vs Alpha')
plt.legend()
plt.show()

In [ ]:
# Best alpha value
best_lasso_alpha_coarse = lasso_alphas[int(np.argmin(lasso_test_scores))]
print('Best alpha (coarse):', best_lasso_alpha_coarse)

# Best score (negative MAE)
print('Best test MAE (coarse):', min(lasso_test_scores))

#### **3.3.3 Alpha Fine-Tuning**

Search a narrower alpha range around the initial best result.

In [ ]:
# List of alphas to tune for Lasso regularization
lasso_alphas_fine = np.linspace(1e-6, 0.005, 30)


In [ ]:
# Tuning Lasso hyperparameters
lasso_train_scores_fine, lasso_test_scores_fine = [], []
for a in lasso_alphas_fine:
    model = Lasso(alpha=a, max_iter=10000)
    model.fit(X_train_scaled, y_train)
    train_mae = mean_absolute_error(y_train, model.predict(X_train_scaled))
    test_mae = mean_absolute_error(y_test, model.predict(X_test_scaled))
    lasso_train_scores_fine.append(train_mae)
    lasso_test_scores_fine.append(test_mae)


In [ ]:
# Plot train and test scores against alpha
plt.figure(figsize=(8, 5))
plt.plot(lasso_alphas_fine, lasso_train_scores_fine, marker='o', label='Train MAE')
plt.plot(lasso_alphas_fine, lasso_test_scores_fine, marker='o', label='Test MAE')
plt.xlabel('Alpha')
plt.ylabel('MAE')
plt.title('Lasso: MAE vs Alpha (fine range)')
plt.legend()
plt.show()


In [ ]:
# Best alpha value
best_lasso_alpha = lasso_alphas_fine[int(np.argmin(lasso_test_scores_fine))]
print('Best alpha (fine):', best_lasso_alpha)

# Best score (negative MAE)
print('Best test MAE (fine):', min(lasso_test_scores_fine))


In [ ]:
# Set best alpha for Lasso regression
lasso_final = Lasso(alpha=best_lasso_alpha, max_iter=10000)

# Fit the Lasso model on scaled training data
lasso_final.fit(X_train_scaled, y_train)

# Get the coefficients of the fitted model
lasso_coefs = pd.Series(lasso_final.coef_, index=X_train_scaled.columns).sort_values()


In [ ]:
# Check the coefficients for each feature
lasso_coefs


In [ ]:
# Evaluate the Lasso model on the test data
lasso_metrics = evaluate(lasso_final, X_train_scaled, y_train, X_test_scaled, y_test, 'Lasso')
lasso_metrics


### **3.4 Regularisation Comparison and Analysis**

#### **3.4.1 Model Performance Comparison**

Compare evaluation metrics across the models.

In [ ]:
# Compare metrics for each model
comparison_df = pd.DataFrame([baseline_metrics, ridge_metrics, lasso_metrics]).set_index('Model')
comparison_df


#### **3.4.2 Coefficient Comparison**

Compare the coefficients learned by the three models.

Also visualise a few of the largest coefficients and the coefficients of features dropped by Lasso.

In [ ]:
# Compare highest coefficients and coefficients of eliminated features
coef_compare = pd.DataFrame({
    'Ridge': ridge_coefs,
    'Lasso': lasso_coefs
})

top_features = coef_compare['Ridge'].abs().sort_values(ascending=False).head(10).index
dropped_by_lasso = coef_compare[coef_compare['Lasso'] == 0].index

print(f'Features eliminated by Lasso: {len(dropped_by_lasso)} of {len(coef_compare)}')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
coef_compare.loc[top_features].plot(kind='barh', ax=axes[0])
axes[0].set_title('Top 10 Coefficients by |Ridge Weight|')

if len(dropped_by_lasso) > 0:
    coef_compare.loc[dropped_by_lasso].head(15)['Ridge'].plot(kind='barh', ax=axes[1], color='orange')
    axes[1].set_title('Ridge Coefficients of Features Dropped by Lasso')
else:
    axes[1].text(0.5, 0.5, 'No features eliminated by Lasso', ha='center', va='center')
    axes[1].set_title('Ridge Coefficients of Features Dropped by Lasso')

plt.tight_layout()
plt.show()


## **4. Conclusion and Key Takeaways**

What did you notice by performing regularisation? Did the model performance improve? If not, then why? Did you find overfitting or not? Was the data sufficent? Is a linear model sufficient?

### **4.1 Findings and Practical Implications**

Summarise the model results, the impact of regularisation, evidence of overfitting, data limitations, and whether a linear model is suitable for this prediction task.